## 1. Data Modeling (world)

### a.

create table language(
<br>    -> id varchar(10) not null,
<br>    -> name varchar(100) not null,
<br>    -> primary key (id));

create table country (
<br>   -> code varchar(10) not null,
<br>   -> name varchar(100) not null,
<br>   -> continent varchar(50) not null,
<br>   -> gnp decimal(10,2),
<br>   -> primary key (code));

create table city (
<br>    -> id int not null,
<br>    -> name varchar(100) not null,
<br>    -> countrycode varchar(10) not null,
<br>    -> population int,
<br>    -> primary key (id),
<br>    -> foreign key (countrycode) references country(code));

create table countrylanguage(
<br>    -> countrycode varchar(10) not null,
<br>    -> language_id varchar(10) not null,
<br>    -> isofficial char(1),
<br>    -> percentage decimal(4, 1),
<br>    -> primary key ( countrycode, language_id)
<br>    -> foreign key (countrycode) references country(code),
<br>    -> foreign key (language_id) references language(id));

### b.

Entity: Country <br>
Attributes: code, name, continent, gnp <br>
Primary Key: code <br>
<br>
Entity: City <br>
Attributes: id, name, population <br>
Primary Key: id <br>
<br>
Entity: Language <br>
Attributes: id, name <br>
Primary Key: id <br>

Relationships: <br>
- City -> Country: Many-to-One
    - Many: City
    - One: Country exactly one
- County <-> Language: Many-to-Many
    - Country: zero or more
    - Language: zero or more


## 2. Parent-Child Schema Analysis (Sakila)

### 1.

select parent_table, count(*) as reference_count
<br>    -> from child_parent
<br>    -> group by parent_table
<br>    -> order by reference_count desc;
<br>

| parent_table | reference_count |
|--------------|-----------------|
| address      | 3               |
| store        | 3               |
| film         | 3               |
| staff        | 3               |
| language     | 2               |
| customer     | 2               |
| city         | 1               |
| country      | 1               |
| actor        | 1               |
| category     | 1               |
| rental       | 1               |
| inventory    | 1               |


### 2.

select child_table, count(parent_table) as dependency_count
<br>    -> from child_parent
<br>    -> group by child_table
<br>    -> having dependency_count > 1
<br>    -> order by dependency_count desc;
<br>

| child_table   | dependency_count    |
|---------------|---------------------|
| payment       | 3                   |
| rental        | 3                   |
| customer      | 2                   |
| film          | 2                   |
| film_actor    | 2                   |
| film_category | 2                   |
| inventory     | 2                   |
| staff         | 2                   |
| store         | 2                   |

### 3.

select distinct(t1.child_table)
<br>    -> from child_parent t1
<br>    -> join child_parent t2 on t1.child_table = t2.parent_table
<br>    -> order by t1.child_table asc;

select child_table
<br>    -> from child_parent
<br>    -> intersect
<br>    -> select parent_table
<br>    -> from child_parent
<br>    -> order by child_table asc;

| child_table |
|-------------|
| address     |
| city        |
| customer    |
| film        |
| inventory   |
| rental      |
| staff       |
| store       |


### 4.

select distinct (t.table_name)
<br>    -> from tbl_columns t
<br>    -> left join child_parent c1 on t.table_name = c1.child_table
<br>    -> left join child_parent c2 on t.table_name = c2.parent_table
<br>    -> where c1.child_table is null and c2.parent_table is null;

select distinct(table_name)
<br>    -> from tbl_columns
<br>    -> except 
<br>    -> select c1.child_table
<br>    -> from child_parent c1
<br>    -> except 
<br>    -> select c2.parent_table
<br>    -> from child_parent c2;

| table_name |
|------------|
| film_text  |

### 5.

select child_table, parent_table, count(*) as num_col
<br>    -> from child_parent
<br>    -> group by child_table, parent_table
<br>    -> having count(*) > 1;

| child_table | parent_table | num_col |
|-------------|--------------|---------|
| film        | language     | 2       |

### 6.

select c.child_table, c.child_col, ct.data_type as child_data_type, c.parent_table, c.parent_col, pt.data_type as parent_data_type
<br>    -> from child_parent c
<br>    -> join tbl_columns ct on c.child_table = ct.table_name and c.child_col = ct.column_name
<br>    -> join tbl_columns pt on c.parent_table = pt.table_name and c.parent_col = pt.column_name;

| child_table   | child_col            | child_data_type | parent_table | parent_col   | parent_data_type |
|---------------|----------------------|-----------------|--------------|--------------|------------------|
| film_actor    | actor_id             | smallint        | actor        | actor_id     | smallint         |
| store         | address_id           | smallint        | address      | address_id   | smallint         |
| staff         | address_id           | smallint        | address      | address_id   | smallint         |
| customer      | address_id           | smallint        | address      | address_id   | smallint         |
| film_category | category_id          | tinyint         | category     | category_id  | tinyint          |
| address       | city_id              | smallint        | city         | city_id      | smallint         |
| city          | country_id           | smallint        | country      | country_id   | smallint         |
| rental        | customer_id          | smallint        | customer     | customer_id  | smallint         |
| payment       | customer_id          | smallint        | customer     | customer_id  | smallint         |
| inventory     | film_id              | smallint        | film         | film_id      | smallint         |
| film_category | film_id              | smallint        | film         | film_id      | smallint         |
| film_actor    | film_id              | smallint        | film         | film_id      | smallint         |
| rental        | inventory_id         | mediumint       | inventory    | inventory_id | mediumint        |
| film          | original_language_id | tinyint         | language     | language_id  | tinyint          |
| film          | language_id          | tinyint         | language     | language_id  | tinyint          |
| payment       | rental_id            | int             | rental       | rental_id    | int              |
| store         | manager_staff_id     | tinyint         | staff        | staff_id     | tinyint          |
| rental        | staff_id             | tinyint         | staff        | staff_id     | tinyint          |
| payment       | staff_id             | tinyint         | staff        | staff_id     | tinyint          |
| staff         | store_id             | tinyint         | store        | store_id     | tinyint          |
| inventory     | store_id             | tinyint         | store        | store_id     | tinyint          |
| customer      | store_id             | tinyint         | store        | store_id     | tinyint          |


### 7.

